# Atividade: construindo regras para classificar frutas

Você receberá exemplos de maçãs, tangerinas, laranjas e limões. Observe os dados e crie suas próprias regras com `if`, `elif` e `else`.

Execute as células na ordem. Não procure o gabarito do teste antes de congelar suas regras. A descoberta do que este processo representa em aprendizado de máquina aparece no final.

## 1. Abrir os exemplos conhecidos

O código abaixo lê um arquivo CSV com Pandas. Cada linha representa uma fruta. As colunas `mass`, `width`, `height` e `color_score` descrevem a fruta; `fruit_name` informa seu nome. Nos dados originais, tangerina aparece como `mandarin`.

In [ ]:
import pandas as pd

URL_TREINO = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/treino.csv'
treino = pd.read_csv(URL_TREINO)
treino.head()

In [ ]:
print(f'Linhas: {len(treino)}')
print('Colunas:', list(treino.columns))
treino['fruit_name'].value_counts()

Antes de programar, observe algumas linhas de cada fruta. Há a mesma quantidade de exemplos para todas as classes? Que consequência isso pode ter?

In [ ]:
treino.groupby('fruit_name')[['mass', 'width', 'height', 'color_score']].agg(['min', 'max'])

Responda antes de continuar:

1. Qual fruta parece mais fácil de reconhecer? Por quê?
2. Quais duas frutas mais se confundem?
3. Uma característica sozinha parece suficiente para todas as decisões?
4. Esboce no papel uma primeira sequência de perguntas do tipo: se isto acontecer, então...

## 2. Escrever as regras

Complete a função abaixo. Ela deve devolver exatamente um destes nomes: `apple`, `mandarin`, `orange` ou `lemon`. Você pode criar quantas condições forem necessárias, mas deve conseguir explicar cada uma delas.

In [ ]:
def classificar_fruta(mass, width, height, color_score):
    # Apague a linha abaixo e escreva suas regras com if, elif e else.
    return 'apple'

# Teste manualmente uma fruta antes de aplicar a função a todas as linhas.
classificar_fruta(mass=192, width=8.4, height=7.3, color_score=0.55)

In [ ]:
def aplicar_regras(dados):
    return dados.apply(
        lambda fruta: classificar_fruta(
            fruta['mass'],
            fruta['width'],
            fruta['height'],
            fruta['color_score'],
        ),
        axis=1,
    )

resultado_treino = treino.copy()
resultado_treino['previsao'] = aplicar_regras(resultado_treino)
resultado_treino['acertou'] = resultado_treino['previsao'] == resultado_treino['fruit_name']
resultado_treino.head()

In [ ]:
acuracia_treino = resultado_treino['acertou'].mean()
print(f'Acurácia nos exemplos conhecidos: {acuracia_treino:.1%}')
display(resultado_treino.loc[~resultado_treino['acertou']])

Ajuste suas regras olhando apenas para esses exemplos. Depois responda:

- Que mudanças aumentaram os acertos?
- Quais erros foram difíceis de corrigir sem criar outro erro?
- Sua função ficou simples de explicar?
- Uma acurácia alta aqui garante que a função acertará frutas novas?

## 3. Congelar as regras

Antes de abrir os exemplos novos:

1. anote sua acurácia nos exemplos conhecidos;
2. estime quantas das 11 frutas novas sua função acertará;
3. registre uma justificativa curta;
4. salve esta versão do notebook e não altere mais a função até avaliar o teste.

**Minha estimativa:** ... acertos em 11.

**Minha justificativa:** ...

## 4. Fazer previsões para frutas novas

O próximo arquivo não contém `fruit_name`. Gere as previsões sem alterar as regras.

In [ ]:
URL_TESTE = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/teste.csv'
teste = pd.read_csv(URL_TESTE)
teste

In [ ]:
previsoes = teste[['id']].copy()
previsoes['previsao'] = aplicar_regras(teste)
previsoes.to_csv('previsoes_frutas.csv', index=False)
display(previsoes)

try:
    from google.colab import files
    files.download('previsoes_frutas.csv')
except ImportError:
    print('O arquivo previsoes_frutas.csv foi salvo na pasta atual.')

## 5. O que você acabou de fazer?

Agora podemos dar nome às partes da experiência:

- massa, largura, altura e medida de cor são as **características** (*features*);
- o nome da fruta é a **classe** ou **rótulo** (*label*);
- sua função de regras é um **modelo**: ela transforma características em uma previsão;
- os exemplos conhecidos formam os **dados de treino**;
- você ajustou o modelo manualmente usando esses dados — fez o papel que, depois, será desempenhado por um algoritmo de treinamento;
- as frutas novas formam os **dados de teste**;
- funcionar em exemplos novos é o que chamamos de **generalização**.

Você ainda não treinou automaticamente um algoritmo de aprendizado de máquina. A atividade separou as etapas para que fosse possível sentir a dificuldade de formular regras e avaliar se elas generalizam.

## 6. Avaliar depois da liberação do gabarito

Cole abaixo o link direto fornecido pelo professor. Execute esta seção somente depois de entregar ou congelar suas previsões.

In [ ]:
URL_GABARITO = ''  # Cole aqui o link direto liberado pelo professor.

if not URL_GABARITO.strip():
    print('O gabarito ainda não foi informado. Preserve suas previsões e volte depois.')
else:
    gabarito = pd.read_csv(URL_GABARITO)
    avaliacao = previsoes.merge(gabarito, on='id', validate='one_to_one')
    avaliacao['acertou'] = avaliacao['previsao'] == avaliacao['fruit_name']

    print(f"Acurácia no teste: {avaliacao['acertou'].mean():.1%}")
    print('\nAcurácia por classe:')
    display(avaliacao.groupby('fruit_name')['acertou'].mean().rename('acuracia'))

    print('Matriz de confusão:')
    display(pd.crosstab(
        avaliacao['fruit_name'],
        avaliacao['previsao'],
        rownames=['classe correta'],
        colnames=['previsão'],
        dropna=False,
    ))

    print('Erros:')
    display(avaliacao.loc[~avaliacao['acertou']])

## 7. Fechamento

Responda com suas palavras:

1. O resultado do teste ficou acima ou abaixo de sua estimativa?
2. Compare a acurácia nos dados de treino com a acurácia no teste. O que mudou?
3. Qual classe teve o pior resultado? A acurácia total deixava isso evidente?
4. Observe os erros. Há frutas com valores fora dos intervalos vistos no treino?
5. O que você mudaria nas regras? Por que não devemos usar o mesmo teste repetidamente para fazer esses ajustes?
6. Que vantagem um algoritmo de treinamento poderia ter sobre o ajuste manual? Que cuidado ainda continuaria necessário?